# CRM Phase 1 — Data Cleaning (Guided)

**Dataset:** `CRM_Raw_Dataset.csv`  
**Approach:** One step at a time. We inspect first, then decide cleaning rules together as a data analyst would.

### Phase 1 roadmap
1. Load & inspect ✓
2. Missing values ✓
3. Duplicates ✓
4. Data types + date validation ✓
5. Text cleaning / category standardization ✓
6. Outliers + invalid / logical checks ✓
7. **Final transforms + export `CRM_Cleaned.csv`** ← we are here

> Work one step at a time. After each step we pause until you say continue.

## Setup — imports

We use `pandas` as the main table tool and `numpy` for numeric helpers. Visualization libraries come later (EDA / dashboard).

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 40)

## Step 1 — Load & inspect

**Why this step?** Before fixing anything, we need a clear picture of:
- how big the dataset is
- what columns we have and their types
- whether values look consistent (case, spaces, missing data)

Skipping this leads to cleaning blindly — fixing symptoms without understanding the data.

### 1.1 Load the raw CSV

Load the file as-is. We do not trim, rename, or cast types yet — that comes after inspection.

In [2]:
df = pd.read_csv("CRM_Raw_Dataset.csv")

print("Loaded successfully.")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

Loaded successfully.
Rows: 5,075 | Columns: 24


### 1.2 Shape, head, and tail

- `shape` → row/column count
- `head()` → first rows (typical values)
- `tail()` → last rows (sometimes export quirks show up here)

In [3]:
print("Shape:", df.shape)
df.head()

Shape: (5075, 24)


,Customer_ID,First_Name,Last_Name,Email,Phone,Region,Customer_Segment,Industry,Lead_Source,Lead_Stage,Customer_Status,Product,Campaign,Preferred_Channel,Assigned_Sales_Rep,Signup_Date,Last_Contact_Date,Lead_Score,Interaction_Count,Website_Visits,Email_Opens,Response_Time_Hours,Customer_Satisfaction,Revenue
0,CUST11730,Rami,Mousa,customer1730@example.com,9.627223e+11,amman,consumer,Technology,instagram,Contacted,INACTIVE,Marketing Add-on,Year-End Campaign,Phone,Ahmad Saleh,2023-02-21,2023-04-24,45.0,2,4,1,12.7,8.2,0.00
1,CUST12656,Rana,Odeh,customer2656@example.com,9.627899e+11,Karak,Small business,Retail,Referral,won,Prospect,CRM Professional,Back-to-School,Phone,Ahmad Saleh,2023-01-13,2023-01-30,69.0,6,13,2,1.2,7.7,1008.38
2,CUST10033,Tareq,Shami,customer33@example.com,9.627317e+11,salt,consumer,Technology,website,won,INACTIVE,Marketing Add-on,Back-to-School,Email,Dana Nasser,2025-01-25,2025-07-25,70.0,7,9,3,6.0,4.8,227.43
3,CUST14927,Yousef,Awad,customer4927@example.com,9.627476e+11,amman,CORPORATE,Technology,website,Contacted,active,Analytics Add-on,No Campaign,Meeting,Yara Odeh,2023-01-25,2023-02-04,42.0,8,6,7,11.8,8.0,0.00
4,CUST13901,Zaid,Najjar,customer3901@example.com,9.627945e+11,Jerash,enterprise,Retail,Google ads,Proposal,active,Marketing Add-on,Q1 Acquisition,WhatsApp,Lina Haddad,2024-09-20,2025-03-15,70.0,2,12,2,9.8,NaN,856.48


In [4]:
df.tail()

,Customer_ID,First_Name,Last_Name,Email,Phone,Region,Customer_Segment,Industry,Lead_Source,Lead_Stage,Customer_Status,Product,Campaign,Preferred_Channel,Assigned_Sales_Rep,Signup_Date,Last_Contact_Date,Lead_Score,Interaction_Count,Website_Visits,Email_Opens,Response_Time_Hours,Customer_Satisfaction,Revenue
5070,CUST14426,Rana,AbuAli,customer4426@example.com,9.627339e+11,Madaba,CORPORATE,Technology,instagram,NEGOTIATION,Prospect,CRM Enterprise,Q1 Acquisition,Email,Ahmad Saleh,2024-05-24,2024-07-28,80.0,4,6,11,21.4,5.8,1302.08
5071,CUST10466,Rana,Jaber,customer466@example.com,9.627814e+11,Madaba,enterprise,Construction,Email Campaign,Contacted,active,CRM Basic,Q1 Acquisition,Website,Yara Odeh,2024-10-25,2025-04-14,59.0,8,8,8,18.8,6.4,0.00
5072,CUST13092,Hala,Najjar,customer3092@example.com,9.627933e+11,IRBID,consumer,Hospitality,LinkedIn,qualified,active,Analytics Add-on,Spring Campaign,Social Media,Lina Haddad,2024-01-29,2024-04-06,60.0,4,5,4,3.9,9.6,0.00
5073,CUST13772,Yara,Nasser,customer3772@example.com,9.627806e+11,IRBID,CORPORATE,Retail,facebook,new lead,active,Analytics Add-on,Q1 Acquisition,Meeting,Dana Nasser,2024-09-16,2025-05-11,49.0,6,15,5,14.8,NaN,0.00
5074,CUST10860,Omar,Jaber,customer860@example.com,NaN,amman,consumer,Healthcare,Cold Call,Contacted,active,CRM Professional,No Campaign,Phone,Samer Khalil,2025-06-21,2025-12-31,52.0,7,9,2,16.6,10.0,0.00


### 1.3 Column list and `info()`

`info()` shows dtype + non-null counts. Gaps between row count and non-null count = missing values we will handle in Step 2.

In [5]:
print("Columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"  {i:2d}. {col}")

print("\n--- info() ---")
df.info()

Columns:
   1. Customer_ID
   2. First_Name
   3. Last_Name
   4. Email
   5. Phone
   6. Region
   7. Customer_Segment
   8. Industry
   9. Lead_Source
  10. Lead_Stage
  11. Customer_Status
  12. Product
  13. Campaign
  14. Preferred_Channel
  15. Assigned_Sales_Rep
  16. Signup_Date
  17. Last_Contact_Date
  18. Lead_Score
  19. Interaction_Count
  20. Website_Visits
  21. Email_Opens
  22. Response_Time_Hours
  23. Customer_Satisfaction
  24. Revenue

--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 5075 entries, 0 to 5074
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Customer_ID            5075 non-null   str    
 1   First_Name             5075 non-null   str    
 2   Last_Name              5075 non-null   str    
 3   Email                  4948 non-null   str    
 4   Phone                  4870 non-null   float64
 5   Region                 4981 non-null   str    
 6   Custome

### 1.4 `describe()` — numeric and categorical overview

- Numeric: min/max/mean help spot impossible values (e.g. negative revenue, extreme response times).
- Object columns: unique counts and top values preview category messiness (mixed case, trailing spaces).

In [6]:
df.describe()

,Phone,Lead_Score,Interaction_Count,Website_Visits,Email_Opens,Response_Time_Hours,Customer_Satisfaction,Revenue
count,4.870000e+03,4950.000000,5075.000000,5075.000000,5075.000000,4898.000000,4798.000000,5075.000000
mean,9.627553e+11,59.420404,5.014778,7.984828,3.986798,11.168232,7.377803,327.492690
std,2.586848e+07,23.149353,2.259648,2.825702,2.000498,7.477474,1.489802,1237.200571
min,9.627100e+11,1.000000,0.000000,0.000000,0.000000,-5.000000,2.200000,-500.000000
25%,9.627334e+11,43.000000,3.000000,6.000000,3.000000,5.700000,6.400000,0.000000
50%,9.627557e+11,60.000000,5.000000,8.000000,4.000000,9.700000,7.400000,0.000000
75%,9.627778e+11,76.000000,6.000000,10.000000,5.000000,15.000000,8.400000,153.935000
max,9.627999e+11,120.000000,14.000000,21.000000,13.000000,61.600000,15.000000,23346.240000


In [7]:
df.describe(include="object")

C:\Users\Omar\AppData\Local\Temp\ipykernel_5152\702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,Customer_ID,First_Name,Last_Name,Email,Region,Customer_Segment,Industry,Lead_Source,Lead_Stage,Customer_Status,Product,Campaign,Preferred_Channel,Assigned_Sales_Rep,Signup_Date,Last_Contact_Date
count,5075,5075,5075,4948,4981,5075,4946,4924,5075,5075,5075,4919,5075,5075,5075,4849
unique,5000,29,26,4875,8,4,8,9,7,3,5,7,6,6,1079,1002
top,CUST14920,Dana,Qasem,customer4920@example.com,amman,consumer,Hospitality,Email Campaign,qualified,active,CRM Professional,No Campaign,Phone,Dana Nasser,2024-01-29,2025-12-31
freq,2,293,312,2,676,2420,646,579,911,3211,1518,943,891,883,12,770


### 1.5 Peek key categoricals (unique values)

We look at raw unique values for important dimensions. This often reveals:
- leading/trailing spaces (`" amman"`)
- inconsistent casing (`won` vs `Won`, `INACTIVE` vs `active`)
- empty strings treated as valid categories

We only **observe** here; standardization is Step 5.

In [8]:
categorical_cols = [
    "Region",
    "Customer_Segment",
    "Industry",
    "Lead_Source",
    "Lead_Stage",
    "Customer_Status",
    "Product",
    "Campaign",
    "Preferred_Channel",
    "Assigned_Sales_Rep",
]

for col in categorical_cols:
    if col not in df.columns:
        print(f"\n⚠ Column missing: {col}")
        continue
    uniques = df[col].dropna().unique()
    # show with repr so spaces/casing are visible
    preview = sorted(uniques, key=lambda x: str(x).lower())
    print(f"\n=== {col} ({len(uniques)} unique, non-null) ===")
    for v in preview:
        print(f"  {v!r}")


=== Region (8 unique, non-null) ===
  ' amman'
  'AQABA'
  'IRBID'
  'Jerash'
  'Karak'
  'Madaba'
  'salt'
  'zarqa '

=== Customer_Segment (4 unique, non-null) ===
  'consumer'
  'CORPORATE'
  'enterprise'
  'Small business '

=== Industry (8 unique, non-null) ===
  'Construction'
  'Education'
  'Finance'
  'Healthcare'
  'Hospitality'
  'Manufacturing'
  'Retail'
  'Technology'

=== Lead_Source (9 unique, non-null) ===
  ' instagram '
  'Cold Call'
  'Email Campaign'
  'facebook'
  'Google ads'
  'LinkedIn'
  'Referral'
  'Trade Show'
  'website'

=== Lead_Stage (7 unique, non-null) ===
  'Contacted'
  'Lost'
  'NEGOTIATION'
  'new lead'
  'Proposal'
  'qualified '
  'won'

=== Customer_Status (3 unique, non-null) ===
  'active'
  'INACTIVE'
  'Prospect'

=== Product (5 unique, non-null) ===
  'Analytics Add-on'
  'CRM Basic'
  'CRM Enterprise'
  'CRM Professional'
  'Marketing Add-on'

=== Campaign (7 unique, non-null) ===
  'Back-to-School'
  'No Campaign'
  'Q1 Acquisition'
  '

### 1.6 Quick missing-value snapshot (preview only)

This is a teaser for Step 2 — counts only, no imputation or drops yet.

In [9]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_pct
Campaign,156,3.07
Customer_Satisfaction,277,5.46
Email,127,2.50
Industry,129,2.54
Last_Contact_Date,226,4.45
Lead_Score,125,2.46
Lead_Source,151,2.98
Phone,205,4.04
Region,94,1.85
Response_Time_Hours,177,3.49


## Step 1 — Analyst observations (no fixes yet)

After running the cells above, expect patterns like these (confirm in your output):

1. **Text inconsistency** — regions/segments/stages often mix case and carry leading/trailing spaces (e.g. `" amman"`, `"Small business "`, `"won"` vs other stages).
2. **Missing contact / score fields** — `Email`, `Phone`, `Lead_Score`, `Customer_Satisfaction`, `Response_Time_Hours`, `Industry`, `Campaign`, `Lead_Source` may have nulls.
3. **Logical questions to park for later** — e.g. `Lead_Stage == "won"` with `Revenue == 0`, or `Customer_Status` vs stage mismatches; we validate these in Step 6.
4. **Dates as object/string** — `Signup_Date` / `Last_Contact_Date` likely need parsing in Step 4.

---

Step 1 complete. Continue below for **Step 2 — Missing values**.

## Step 2 — Missing values

**Why this step?** Missing data changes every KPI (conversion, average satisfaction, revenue by region). We inventory first, then apply **explicit rules** so the dashboard stays honest.

We work on a copy `df_clean` so the raw `df` stays available for comparison.

### 2.1 Full missing-value inventory

Counts + % for every column that has nulls. Also check whitespace-only strings (they look filled but are empty).

In [10]:
df_clean = df.copy()

missing = df_clean.isna().sum()
missing_pct = (df_clean.isna().mean() * 100).round(2)
missing_inventory = (
    pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)

print(f"Columns with NaN: {len(missing_inventory)} / {df_clean.shape[1]}")
print(f"Total NaN cells: {int(missing.sum()):,}")
missing_inventory

Columns with NaN: 10 / 24
Total NaN cells: 1,667


,missing_count,missing_pct
Customer_Satisfaction,277,5.46
Last_Contact_Date,226,4.45
Phone,205,4.04
Response_Time_Hours,177,3.49
Campaign,156,3.07
Lead_Source,151,2.98
Industry,129,2.54
Email,127,2.50
Lead_Score,125,2.46
Region,94,1.85


In [11]:
# Whitespace-only / blank strings (not NaN, but still "missing" for analysis)
blank_rows = []
for col in df_clean.select_dtypes(include=["object", "string"]).columns:
    blank_mask = df_clean[col].notna() & df_clean[col].astype(str).str.strip().eq("")
    n_blank = int(blank_mask.sum())
    if n_blank > 0:
        blank_rows.append({"column": col, "blank_string_count": n_blank})

blank_strings = pd.DataFrame(blank_rows)
if blank_strings.empty:
    print("No whitespace-only blank strings found in object columns.")
else:
    blank_strings

No whitespace-only blank strings found in object columns.


### 2.2 Handling rules (analyst decisions)

| Column group | Columns | Rule | Why |
|---|---|---|---|
| Business categories | `Region`, `Industry`, `Lead_Source`, `Campaign` | Fill NaN with `"Unknown"` | Keeps rows in groupbys/charts instead of silently dropping them |
| Contact fields | `Email`, `Phone` | **Leave as NaN** | Inventing emails/phones is dishonest; contact completeness can be measured later |
| Survey / ops metrics | `Lead_Score`, `Customer_Satisfaction`, `Response_Time_Hours` | **Leave as NaN** | Mean/median stay undistorted; we report “known responses only” |
| Dates | `Last_Contact_Date` | **Leave as NaN** | Dates are parsed in Step 4; we never invent a contact date |
| Row drops | — | **Do not drop rows** for missing values in this step | Missing rate is low (~1–5% per column); dropping would bias the funnel |

> Text trim / category casing is **Step 5**, not here.

### 2.3 Apply category fill (`Unknown`)

Only the four business category columns change in this step.

In [12]:
category_fill_cols = ["Region", "Industry", "Lead_Source", "Campaign"]

before = df_clean[category_fill_cols].isna().sum()

for col in category_fill_cols:
    df_clean[col] = df_clean[col].fillna("Unknown")

after = df_clean[category_fill_cols].isna().sum()

fill_report = pd.DataFrame({
    "missing_before": before,
    "missing_after": after,
    "filled_with_Unknown": before - after,
})
fill_report

,missing_before,missing_after,filled_with_Unknown
Region,94,0,94
Industry,129,0,129
Lead_Source,151,0,151
Campaign,156,0,156


### 2.4 Verify remaining missing values

These remaining nulls are intentional (contacts, scores, dates).

In [13]:
remaining = df_clean.isna().sum()
remaining_pct = (df_clean.isna().mean() * 100).round(2)
remaining_missing = (
    pd.DataFrame({"missing_count": remaining, "missing_pct": remaining_pct})
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)

print(f"Rows unchanged: {len(df_clean):,} (same as raw {len(df):,})")
print("Category fills applied. Remaining NaNs are kept on purpose:")
remaining_missing

Rows unchanged: 5,075 (same as raw 5,075)
Category fills applied. Remaining NaNs are kept on purpose:


,missing_count,missing_pct
Customer_Satisfaction,277,5.46
Last_Contact_Date,226,4.45
Phone,205,4.04
Response_Time_Hours,177,3.49
Email,127,2.50
Lead_Score,125,2.46


## Step 2 — Summary

**Done**
- Inventoried NaNs (and checked for blank strings)
- Filled `Region`, `Industry`, `Lead_Source`, `Campaign` → `"Unknown"`
- Left `Email`, `Phone`, `Lead_Score`, `Customer_Satisfaction`, `Response_Time_Hours`, `Last_Contact_Date` as NaN
- No rows dropped

**Working frame:** `df_clean` (raw `df` unchanged)

---

Step 2 complete. Continue below for **Step 3 — Duplicates**.

## Step 3 — Duplicates

**Why this step?** Duplicate customers inflate lead counts, conversion rates, and revenue totals. We check:
1. Exact full-row duplicates
2. Duplicate `Customer_ID` (business key)
3. Near-duplicates (same ID, different field values)

### 3.1 Exact full-row duplicates

`duplicated()` marks every repeated copy after the first. We also show both copies with `keep=False` for review.

In [14]:
n_before = len(df_clean)
exact_dupe_mask = df_clean.duplicated(keep="first")
n_exact_dupes = int(exact_dupe_mask.sum())

print(f"Rows before: {n_before:,}")
print(f"Exact full-row duplicate rows to remove: {n_exact_dupes}")

# Preview pairs (both original + copy)
exact_dupe_preview = df_clean[df_clean.duplicated(keep=False)].sort_values("Customer_ID")
print(f"Rows involved in exact-dupe groups: {len(exact_dupe_preview)}")
exact_dupe_preview.head(10)

Rows before: 5,075
Exact full-row duplicate rows to remove: 74
Rows involved in exact-dupe groups: 148


,Customer_ID,First_Name,Last_Name,Email,Phone,Region,Customer_Segment,Industry,Lead_Source,Lead_Stage,Customer_Status,Product,Campaign,Preferred_Channel,Assigned_Sales_Rep,Signup_Date,Last_Contact_Date,Lead_Score,Interaction_Count,Website_Visits,Email_Opens,Response_Time_Hours,Customer_Satisfaction,Revenue
4117,CUST10029,Dana,Dahleh,customer29@example.com,9.627583e+11,Unknown,consumer,Finance,LinkedIn,new lead,Prospect,CRM Enterprise,Q1 Acquisition,Website,Omar Mansour,2024-08-28,2025-05-02,12.0,3,7,3,6.2,6.5,0.00
140,CUST10029,Dana,Dahleh,customer29@example.com,9.627583e+11,Unknown,consumer,Finance,LinkedIn,new lead,Prospect,CRM Enterprise,Q1 Acquisition,Website,Omar Mansour,2024-08-28,2025-05-02,12.0,3,7,3,6.2,6.5,0.00
3371,CUST10088,Khaled,Mansour,customer88@example.com,9.627384e+11,Unknown,consumer,Manufacturing,Google ads,qualified,active,Analytics Add-on,Spring Campaign,Meeting,Yara Odeh,2025-11-07,2025-12-30,50.0,9,10,5,6.1,6.9,0.00
250,CUST10088,Khaled,Mansour,customer88@example.com,9.627384e+11,Unknown,consumer,Manufacturing,Google ads,qualified,active,Analytics Add-on,Spring Campaign,Meeting,Yara Odeh,2025-11-07,2025-12-30,50.0,9,10,5,6.1,6.9,0.00
3400,CUST10245,Layla,Barakat,customer245@example.com,9.627493e+11,Madaba,Small business,Unknown,Google ads,NEGOTIATION,Prospect,Marketing Add-on,No Campaign,Social Media,Lina Haddad,2025-09-29,2025-12-31,76.0,5,7,3,15.5,7.9,320.71
1712,CUST10245,Layla,Barakat,customer245@example.com,9.627493e+11,Madaba,Small business,Unknown,Google ads,NEGOTIATION,Prospect,Marketing Add-on,No Campaign,Social Media,Lina Haddad,2025-09-29,2025-12-31,76.0,5,7,3,15.5,7.9,320.71
2873,CUST10274,Dana,Qasem,customer274@example.com,9.627233e+11,amman,consumer,Finance,Google ads,NEGOTIATION,active,CRM Professional,Back-to-School,Email,Dana Nasser,2023-12-25,2024-01-04,52.0,10,4,4,18.0,8.5,85.56
839,CUST10274,Dana,Qasem,customer274@example.com,9.627233e+11,amman,consumer,Finance,Google ads,NEGOTIATION,active,CRM Professional,Back-to-School,Email,Dana Nasser,2023-12-25,2024-01-04,52.0,10,4,4,18.0,8.5,85.56
95,CUST10346,Dana,Qasem,customer346@example.com,9.627644e+11,zarqa,Small business,Manufacturing,Cold Call,Contacted,active,CRM Professional,Referral Program,WhatsApp,Yara Odeh,2024-04-05,2024-06-17,28.0,4,7,2,7.8,5.0,0.00
463,CUST10346,Dana,Qasem,customer346@example.com,9.627644e+11,zarqa,Small business,Manufacturing,Cold Call,Contacted,active,CRM Professional,Referral Program,WhatsApp,Yara Odeh,2024-04-05,2024-06-17,28.0,4,7,2,7.8,5.0,0.00


### 3.2 Duplicate `Customer_ID` (before dropping)

`Customer_ID` should be unique. If ID dupes > exact full-row dupes, we have conflicting near-duplicates.

In [15]:
id_dupe_count = int(df_clean["Customer_ID"].duplicated().sum())
print(f"Duplicate Customer_ID rows (extra copies): {id_dupe_count}")
print(f"Unique Customer_ID: {df_clean['Customer_ID'].nunique():,} / {len(df_clean):,} rows")

id_dupe_preview = (
    df_clean[df_clean["Customer_ID"].duplicated(keep=False)]
    .sort_values("Customer_ID")
)
id_dupe_preview.head(10)

Duplicate Customer_ID rows (extra copies): 75
Unique Customer_ID: 5,000 / 5,075 rows


,Customer_ID,First_Name,Last_Name,Email,Phone,Region,Customer_Segment,Industry,Lead_Source,Lead_Stage,Customer_Status,Product,Campaign,Preferred_Channel,Assigned_Sales_Rep,Signup_Date,Last_Contact_Date,Lead_Score,Interaction_Count,Website_Visits,Email_Opens,Response_Time_Hours,Customer_Satisfaction,Revenue
140,CUST10029,Dana,Dahleh,customer29@example.com,9.627583e+11,Unknown,consumer,Finance,LinkedIn,new lead,Prospect,CRM Enterprise,Q1 Acquisition,Website,Omar Mansour,2024-08-28,2025-05-02,12.0,3,7,3,6.2,6.5,0.00
4117,CUST10029,Dana,Dahleh,customer29@example.com,9.627583e+11,Unknown,consumer,Finance,LinkedIn,new lead,Prospect,CRM Enterprise,Q1 Acquisition,Website,Omar Mansour,2024-08-28,2025-05-02,12.0,3,7,3,6.2,6.5,0.00
3371,CUST10088,Khaled,Mansour,customer88@example.com,9.627384e+11,Unknown,consumer,Manufacturing,Google ads,qualified,active,Analytics Add-on,Spring Campaign,Meeting,Yara Odeh,2025-11-07,2025-12-30,50.0,9,10,5,6.1,6.9,0.00
250,CUST10088,Khaled,Mansour,customer88@example.com,9.627384e+11,Unknown,consumer,Manufacturing,Google ads,qualified,active,Analytics Add-on,Spring Campaign,Meeting,Yara Odeh,2025-11-07,2025-12-30,50.0,9,10,5,6.1,6.9,0.00
1712,CUST10245,Layla,Barakat,customer245@example.com,9.627493e+11,Madaba,Small business,Unknown,Google ads,NEGOTIATION,Prospect,Marketing Add-on,No Campaign,Social Media,Lina Haddad,2025-09-29,2025-12-31,76.0,5,7,3,15.5,7.9,320.71
3400,CUST10245,Layla,Barakat,customer245@example.com,9.627493e+11,Madaba,Small business,Unknown,Google ads,NEGOTIATION,Prospect,Marketing Add-on,No Campaign,Social Media,Lina Haddad,2025-09-29,2025-12-31,76.0,5,7,3,15.5,7.9,320.71
2873,CUST10274,Dana,Qasem,customer274@example.com,9.627233e+11,amman,consumer,Finance,Google ads,NEGOTIATION,active,CRM Professional,Back-to-School,Email,Dana Nasser,2023-12-25,2024-01-04,52.0,10,4,4,18.0,8.5,85.56
839,CUST10274,Dana,Qasem,customer274@example.com,9.627233e+11,amman,consumer,Finance,Google ads,NEGOTIATION,active,CRM Professional,Back-to-School,Email,Dana Nasser,2023-12-25,2024-01-04,52.0,10,4,4,18.0,8.5,85.56
95,CUST10346,Dana,Qasem,customer346@example.com,9.627644e+11,zarqa,Small business,Manufacturing,Cold Call,Contacted,active,CRM Professional,Referral Program,WhatsApp,Yara Odeh,2024-04-05,2024-06-17,28.0,4,7,2,7.8,5.0,0.00
463,CUST10346,Dana,Qasem,customer346@example.com,9.627644e+11,zarqa,Small business,Manufacturing,Cold Call,Contacted,active,CRM Professional,Referral Program,WhatsApp,Yara Odeh,2024-04-05,2024-06-17,28.0,4,7,2,7.8,5.0,0.00


### 3.3 Handling rules

| Case | Rule | Why |
|---|---|---|
| Exact full-row duplicate | Keep first, drop the rest | Same record exported twice — no information loss |
| Same `Customer_ID`, different values | Keep the row with the more valid `Response_Time_Hours` (non-negative preferred); if tie, keep first | Near-dupe found in this dataset differs on response time; negative hours are invalid (Step 6 will reinforce this) |

We apply these now so each customer appears once.

In [16]:
# 1) Drop exact full-row duplicates
rows_before_exact = len(df_clean)
df_clean = df_clean.drop_duplicates(keep="first").copy()
removed_exact = rows_before_exact - len(df_clean)
print(f"Removed exact duplicates: {removed_exact}")
print(f"Rows after exact de-dupe: {len(df_clean):,}")

# 2) Resolve remaining Customer_ID conflicts (near-duplicates)
conflict_ids = df_clean.loc[df_clean["Customer_ID"].duplicated(keep=False), "Customer_ID"].unique()
print(f"Conflicting Customer_IDs remaining: {len(conflict_ids)}")

if len(conflict_ids) > 0:
    display_cols = ["Customer_ID", "Response_Time_Hours", "Revenue", "Lead_Stage", "Email"]
    conflict_view = df_clean[df_clean["Customer_ID"].isin(conflict_ids)].sort_values("Customer_ID")
    print("Conflict preview:")
    print(conflict_view[display_cols].to_string())

    # Prefer non-negative (or NaN) response time over negative; then higher value
    df_clean["_rt_valid"] = (
        df_clean["Response_Time_Hours"].isna()
        | (df_clean["Response_Time_Hours"] >= 0)
    )
    df_clean = (
        df_clean.sort_values(
            by=["Customer_ID", "_rt_valid", "Response_Time_Hours"],
            ascending=[True, False, False],
            na_position="last",
            kind="mergesort",
        )
        .drop_duplicates(subset=["Customer_ID"], keep="first")
        .drop(columns="_rt_valid")
        .reset_index(drop=True)
    )
else:
    df_clean = df_clean.reset_index(drop=True)

print(f"\nRows after conflict resolve: {len(df_clean):,}")
print(f"Customer_ID unique: {df_clean['Customer_ID'].nunique():,}")
print(f"Remaining Customer_ID dupes: {int(df_clean['Customer_ID'].duplicated().sum())}")
print(f"Remaining exact full-row dupes: {int(df_clean.duplicated().sum())}")

Removed exact duplicates: 74
Rows after exact de-dupe: 5,001
Conflicting Customer_IDs remaining: 1
Conflict preview:
     Customer_ID  Response_Time_Hours  Revenue Lead_Stage                     Email
2393   CUST11885                 31.9   158.37   Proposal  customer1885@example.com
3258   CUST11885                 -5.0   158.37   Proposal  customer1885@example.com



Rows after conflict resolve: 5,000
Customer_ID unique: 5,000
Remaining Customer_ID dupes: 0
Remaining exact full-row dupes: 0


## Step 3 — Summary

**Done**
- Found ~74 exact full-row duplicates → removed (keep first)
- Found 1 near-duplicate `Customer_ID` (`CUST11885`) differing on `Response_Time_Hours` (`31.9` vs `-5.0`) → kept the non-negative value
- Final frame: **5,000 unique customers**, zero ID / full-row duplicates

**Working frame:** `df_clean`

---

Step 3 complete. Continue below for **Step 4 — Data types + date validation**.

## Step 4 — Data types + date validation

**Why this step?** Wrong dtypes break filters, time intelligence, and joins later.
- Dates stored as text → cannot compute tenure / response lag
- Phone as `float64` → loses leading zeros / shows scientific notation
- Impossible date order (`Last_Contact` before `Signup`) → corrupts funnel timing metrics

### 4.1 Current dtypes (before casting)

In [17]:
print(df_clean.dtypes)
print("\nDate columns (sample):")
df_clean[["Signup_Date", "Last_Contact_Date"]].head()

Customer_ID                  str
First_Name                   str
Last_Name                    str
Email                        str
Phone                    float64
Region                       str
Customer_Segment             str
Industry                     str
Lead_Source                  str
Lead_Stage                   str
Customer_Status              str
Product                      str
Campaign                     str
Preferred_Channel            str
Assigned_Sales_Rep           str
Signup_Date                  str
Last_Contact_Date            str
Lead_Score               float64
Interaction_Count          int64
Website_Visits             int64
Email_Opens                int64
Response_Time_Hours      float64
Customer_Satisfaction    float64
Revenue                  float64
dtype: object

Date columns (sample):


,Signup_Date,Last_Contact_Date
0,2025-01-30,2025-03-20
1,2024-10-18,2025-02-20
2,2024-10-20,2024-11-02
3,2023-09-02,2023-10-08
4,2023-06-04,2023-09-13


### 4.2 Parse dates + validation checks

Rules we will apply after the audit:
1. Convert `Signup_Date` / `Last_Contact_Date` with `to_datetime(errors="coerce")`
2. If `Last_Contact_Date < Signup_Date` → **swap** the two dates (pattern looks like transposed entry; both values are otherwise valid)
3. Flag dates after the analysis reference date (should be rare / zero)
4. Convert `Phone` from float → nullable string (no invented numbers; NaN stays missing)

In [18]:
ANALYSIS_DATE = pd.Timestamp("2026-08-12")  # fixed reference for reproducibility

signup_raw = pd.to_datetime(df_clean["Signup_Date"], errors="coerce")
last_raw = pd.to_datetime(df_clean["Last_Contact_Date"], errors="coerce")

parse_fail_signup = int(signup_raw.isna().sum() - df_clean["Signup_Date"].isna().sum())
parse_fail_last = int(last_raw.isna().sum() - df_clean["Last_Contact_Date"].isna().sum())
bad_order = (last_raw < signup_raw) & last_raw.notna() & signup_raw.notna()
future_signup = signup_raw > ANALYSIS_DATE
future_last = last_raw > ANALYSIS_DATE

date_audit = pd.DataFrame({
    "check": [
        "Signup parse failures",
        "Last_Contact parse failures",
        "Last_Contact < Signup",
        "Signup after analysis date",
        "Last_Contact after analysis date",
        "Signup min",
        "Signup max",
        "Last_Contact min",
        "Last_Contact max",
    ],
    "result": [
        parse_fail_signup,
        parse_fail_last,
        int(bad_order.sum()),
        int(future_signup.sum()),
        int(future_last.sum()),
        str(signup_raw.min().date()) if signup_raw.notna().any() else None,
        str(signup_raw.max().date()) if signup_raw.notna().any() else None,
        str(last_raw.min().date()) if last_raw.notna().any() else None,
        str(last_raw.max().date()) if last_raw.notna().any() else None,
    ],
})
date_audit

,check,result
0,Signup parse failures,0
1,Last_Contact parse failures,0
2,Last_Contact < Signup,20
3,Signup after analysis date,0
4,Last_Contact after analysis date,0
5,Signup min,2023-01-01
6,Signup max,2025-12-30
7,Last_Contact min,2023-01-12
8,Last_Contact max,2025-12-31


### 4.3 Apply date casting + swap invalid order

Preview the 20 inverted pairs, then swap and assign datetime dtypes.

In [19]:
# Preview inverted pairs before fix
preview_bad = df_clean.loc[
    bad_order,
    ["Customer_ID", "Signup_Date", "Last_Contact_Date", "Lead_Stage"],
].copy()
preview_bad["Signup_Date"] = signup_raw[bad_order].dt.date
preview_bad["Last_Contact_Date"] = last_raw[bad_order].dt.date
print(f"Inverted date rows to swap: {len(preview_bad)}")
preview_bad.head(10)

Inverted date rows to swap: 20


,Customer_ID,Signup_Date,Last_Contact_Date,Lead_Stage
302,CUST10302,2025-07-08,2025-06-08,Proposal
803,CUST10803,2025-04-10,2025-03-11,qualified
1028,CUST11028,2024-12-19,2024-11-19,won
1358,CUST11358,2023-05-13,2023-04-13,Contacted
1659,CUST11659,2025-09-06,2025-08-07,new lead
1931,CUST11931,2025-03-20,2025-02-18,new lead
1971,CUST11971,2025-10-13,2025-09-13,new lead
2183,CUST12183,2025-05-29,2025-04-29,Contacted
2578,CUST12578,2023-03-04,2023-02-02,won
2710,CUST12710,2025-12-16,2025-11-16,new lead


In [20]:
signup_fixed = signup_raw.copy()
last_fixed = last_raw.copy()

# Swap where last contact is before signup
signup_fixed.loc[bad_order] = last_raw.loc[bad_order]
last_fixed.loc[bad_order] = signup_raw.loc[bad_order]

df_clean["Signup_Date"] = signup_fixed
df_clean["Last_Contact_Date"] = last_fixed

# Verify
still_bad = (
    (df_clean["Last_Contact_Date"] < df_clean["Signup_Date"])
    & df_clean["Last_Contact_Date"].notna()
    & df_clean["Signup_Date"].notna()
)
print(f"Swapped rows: {int(bad_order.sum())}")
print(f"Remaining Last < Signup: {int(still_bad.sum())}")
print(f"Signup_Date dtype: {df_clean['Signup_Date'].dtype}")
print(f"Last_Contact_Date dtype: {df_clean['Last_Contact_Date'].dtype}")
print(
    "Date range Signup:",
    df_clean["Signup_Date"].min().date(),
    "→",
    df_clean["Signup_Date"].max().date(),
)

Swapped rows: 20
Remaining Last < Signup: 0
Signup_Date dtype: datetime64[us]
Last_Contact_Date dtype: datetime64[us]
Date range Signup: 2023-01-01 → 2025-12-30


### 4.4 Fix `Phone` dtype (float → string)

Phones were read as `float64` because of missing values. We convert to nullable string digits (e.g. `"9627..."`), keeping missing as `<NA>`.

In [21]:
phone_before_dtype = df_clean["Phone"].dtype
phone_missing_before = int(df_clean["Phone"].isna().sum())

df_clean["Phone"] = (
    df_clean["Phone"]
    .astype("Int64")          # nullable integer, drops .0
    .astype("string")         # proper string dtype
)

print(f"Phone dtype: {phone_before_dtype} → {df_clean['Phone'].dtype}")
print(f"Phone missing (unchanged): {phone_missing_before} → {int(df_clean['Phone'].isna().sum())}")
df_clean[["Customer_ID", "Phone"]].head()

Phone dtype: float64 → string
Phone missing (unchanged): 200 → 200


,Customer_ID,Phone
0,CUST10000,962716349958
1,CUST10001,962753765818
2,CUST10002,962784357962
3,CUST10003,962784969084
4,CUST10004,962796712050


### 4.5 Confirm numeric columns stay numeric

Counts/scores/revenue should remain numeric for KPIs. We only report dtypes here (outlier rules come in Step 6).

In [22]:
numeric_cols = [
    "Lead_Score",
    "Interaction_Count",
    "Website_Visits",
    "Email_Opens",
    "Response_Time_Hours",
    "Customer_Satisfaction",
    "Revenue",
]

print(df_clean[numeric_cols].dtypes)
print("\nFull dtypes after Step 4:")
print(df_clean.dtypes)

Lead_Score               float64
Interaction_Count          int64
Website_Visits             int64
Email_Opens                int64
Response_Time_Hours      float64
Customer_Satisfaction    float64
Revenue                  float64
dtype: object

Full dtypes after Step 4:
Customer_ID                         str
First_Name                          str
Last_Name                           str
Email                               str
Phone                            string
Region                              str
Customer_Segment                    str
Industry                            str
Lead_Source                         str
Lead_Stage                          str
Customer_Status                     str
Product                             str
Campaign                            str
Preferred_Channel                   str
Assigned_Sales_Rep                  str
Signup_Date              datetime64[us]
Last_Contact_Date        datetime64[us]
Lead_Score                      float64
Interact

## Step 4 — Summary

**Done**
- Parsed `Signup_Date` / `Last_Contact_Date` → datetime (0 parse failures)
- Swapped **20** rows where `Last_Contact < Signup` (transposed-looking pairs)
- No dates after analysis reference `2026-08-12`
- Converted `Phone` float → nullable string
- Left score/revenue numerics as-is for Step 6

**Working frame:** `df_clean`

---

Step 4 complete. Continue below for **Step 5 — Text cleaning / category standardization**.

## Step 5 — Text cleaning / category standardization

**Why this step?** Dashboards group by text labels. `" amman"`, `"Amman"`, and `"AMMAN"` would appear as three regions and split revenue incorrectly.

We will:
1. Strip whitespace on all text columns
2. Normalize casing with explicit maps for messy categories
3. Title-case names
4. Lowercase emails (standard contact hygiene)

### 5.1 Before — unique values that need cleanup

Focus on columns with spaces / mixed case.

In [23]:
messy_cols = [
    "Region",
    "Customer_Segment",
    "Lead_Source",
    "Lead_Stage",
    "Customer_Status",
    "Last_Name",
]

for col in messy_cols:
    uniques = sorted(df_clean[col].dropna().astype(str).unique(), key=str.lower)
    print(f"\n=== {col} ({len(uniques)} unique) ===")
    for v in uniques:
        print(f"  {v!r}")


=== Region (9 unique) ===
  ' amman'
  'AQABA'
  'IRBID'
  'Jerash'
  'Karak'
  'Madaba'
  'salt'
  'Unknown'
  'zarqa '

=== Customer_Segment (4 unique) ===
  'consumer'
  'CORPORATE'
  'enterprise'
  'Small business '

=== Lead_Source (10 unique) ===
  ' instagram '
  'Cold Call'
  'Email Campaign'
  'facebook'
  'Google ads'
  'LinkedIn'
  'Referral'
  'Trade Show'
  'Unknown'
  'website'

=== Lead_Stage (7 unique) ===
  'Contacted'
  'Lost'
  'NEGOTIATION'
  'new lead'
  'Proposal'
  'qualified '
  'won'

=== Customer_Status (3 unique) ===
  'active'
  'INACTIVE'
  'Prospect'

=== Last_Name (26 unique) ===
  ' AbuAli '
  ' Barakat '
  ' Dahleh '
  ' Hamdan '
  ' Mansour '
  ' Odeh '
  ' Saad '
  ' Shami '
  'AbuAli'
  'Awad'
  'Barakat'
  'Dahleh'
  'Haddad'
  'Hamdan'
  'Hassan'
  'Jaber'
  'Khalil'
  'Mansour'
  'Mousa'
  'Najjar'
  'Nasser'
  'Odeh'
  'Qasem'
  'Saad'
  'Saleh'
  'Shami'


### 5.2 Standardization rules

| Column | Rule |
|---|---|
| All text | `.str.strip()` |
| `Region` | Title Case (`Amman`, `Aqaba`, `Irbid`, …); keep `Unknown` |
| `Customer_Segment` | `Consumer`, `Corporate`, `Enterprise`, `Small Business` |
| `Lead_Source` | Title Case + fix brands (`Instagram`, `Facebook`, `Website`, `Google Ads`) |
| `Lead_Stage` | `New Lead`, `Qualified`, `Contacted`, `Proposal`, `Negotiation`, `Won`, `Lost` |
| `Customer_Status` | `Active`, `Inactive`, `Prospect` |
| `Industry`, `Product`, `Campaign`, `Preferred_Channel`, `Assigned_Sales_Rep` | Strip only (already consistent) |
| `First_Name`, `Last_Name` | Strip; Title Case only if ALL CAPS / all lowercase (keep `AbuAli`) |
| `Email` | Strip + lowercase (leave `<NA>` as missing) |

In [24]:
def clean_text_series(s: pd.Series) -> pd.Series:
    """Strip whitespace; preserve nulls."""
    out = s.astype("string").str.strip()
    return out


def normalize_person_name(val):
    """Title-case ALL CAPS / all lowercase; keep mixed case (e.g. AbuAli)."""
    if pd.isna(val):
        return val
    s = str(val).strip()
    if s == "":
        return pd.NA
    if s.isupper() or s.islower():
        return s.title()
    return s


# --- strip all text-like columns first ---
text_cols = [
    "Customer_ID",
    "First_Name",
    "Last_Name",
    "Email",
    "Phone",
    "Region",
    "Customer_Segment",
    "Industry",
    "Lead_Source",
    "Lead_Stage",
    "Customer_Status",
    "Product",
    "Campaign",
    "Preferred_Channel",
    "Assigned_Sales_Rep",
]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = clean_text_series(df_clean[col])

# --- explicit category maps (keys are lowercased stripped values) ---
region_map = {
    "amman": "Amman",
    "aqaba": "Aqaba",
    "irbid": "Irbid",
    "jerash": "Jerash",
    "karak": "Karak",
    "madaba": "Madaba",
    "salt": "Salt",
    "zarqa": "Zarqa",
    "unknown": "Unknown",
}

segment_map = {
    "consumer": "Consumer",
    "corporate": "Corporate",
    "enterprise": "Enterprise",
    "small business": "Small Business",
}

lead_source_map = {
    "instagram": "Instagram",
    "facebook": "Facebook",
    "website": "Website",
    "google ads": "Google Ads",
    "linkedin": "LinkedIn",
    "cold call": "Cold Call",
    "email campaign": "Email Campaign",
    "referral": "Referral",
    "trade show": "Trade Show",
    "unknown": "Unknown",
}

lead_stage_map = {
    "new lead": "New Lead",
    "qualified": "Qualified",
    "contacted": "Contacted",
    "proposal": "Proposal",
    "negotiation": "Negotiation",
    "won": "Won",
    "lost": "Lost",
}

status_map = {
    "active": "Active",
    "inactive": "Inactive",
    "prospect": "Prospect",
}


def apply_map(series: pd.Series, mapping: dict) -> pd.Series:
    key = series.astype("string").str.strip().str.lower()
    mapped = key.map(mapping)
    # If anything unmapped appears, keep Title Case fallback so we notice it
    fallback = series.astype("string").str.strip().str.title()
    return mapped.fillna(fallback)


df_clean["Region"] = apply_map(df_clean["Region"], region_map)
df_clean["Customer_Segment"] = apply_map(df_clean["Customer_Segment"], segment_map)
df_clean["Lead_Source"] = apply_map(df_clean["Lead_Source"], lead_source_map)
df_clean["Lead_Stage"] = apply_map(df_clean["Lead_Stage"], lead_stage_map)
df_clean["Customer_Status"] = apply_map(df_clean["Customer_Status"], status_map)

df_clean["First_Name"] = df_clean["First_Name"].map(normalize_person_name).astype("string")
df_clean["Last_Name"] = df_clean["Last_Name"].map(normalize_person_name).astype("string")

# Email: lowercase when present
email_mask = df_clean["Email"].notna()
df_clean.loc[email_mask, "Email"] = (
    df_clean.loc[email_mask, "Email"].astype(str).str.lower()
)

print("Step 5 transforms applied.")

Step 5 transforms applied.

### 5.3 After — verify cleaned categories

In [25]:
verify_cols = [
    "Region",
    "Customer_Segment",
    "Lead_Source",
    "Lead_Stage",
    "Customer_Status",
    "Industry",
    "Product",
    "Campaign",
    "Preferred_Channel",
    "Assigned_Sales_Rep",
]

for col in verify_cols:
    uniques = sorted(df_clean[col].dropna().astype(str).unique(), key=str.lower)
    print(f"\n=== {col} ({len(uniques)} unique) ===")
    for v in uniques:
        print(f"  {v!r}")

print("\nName / email sample:")
df_clean[["First_Name", "Last_Name", "Email", "Region", "Lead_Stage"]].head(10)


=== Region (9 unique) ===
  'Amman'
  'Aqaba'
  'Irbid'
  'Jerash'
  'Karak'
  'Madaba'
  'Salt'
  'Unknown'
  'Zarqa'

=== Customer_Segment (4 unique) ===
  'Consumer'
  'Corporate'
  'Enterprise'
  'Small Business'

=== Lead_Source (10 unique) ===
  'Cold Call'
  'Email Campaign'
  'Facebook'
  'Google Ads'
  'Instagram'
  'LinkedIn'
  'Referral'
  'Trade Show'
  'Unknown'
  'Website'

=== Lead_Stage (7 unique) ===
  'Contacted'
  'Lost'
  'Negotiation'
  'New Lead'
  'Proposal'
  'Qualified'
  'Won'

=== Customer_Status (3 unique) ===
  'Active'
  'Inactive'
  'Prospect'

=== Industry (9 unique) ===
  'Construction'
  'Education'
  'Finance'
  'Healthcare'
  'Hospitality'
  'Manufacturing'
  'Retail'
  'Technology'
  'Unknown'

=== Product (5 unique) ===
  'Analytics Add-on'
  'CRM Basic'
  'CRM Enterprise'
  'CRM Professional'
  'Marketing Add-on'

=== Campaign (8 unique) ===
  'Back-to-School'
  'No Campaign'
  'Q1 Acquisition'
  'Referral Program'
  'Spring Campaign'
  'Summer C

,First_Name,Last_Name,Email,Region,Lead_Stage
0,Noor,Hassan,customer0@example.com,Aqaba,New Lead
1,Dana,Odeh,customer1@example.com,Aqaba,Contacted
2,Yara,Khalil,customer2@example.com,Salt,Proposal
3,Yara,Saleh,customer3@example.com,Karak,Qualified
4,Zaid,Hamdan,customer4@example.com,Salt,Qualified
5,Yara,Saleh,customer5@example.com,Salt,Contacted
6,Omar,Dahleh,customer6@example.com,Madaba,Negotiation
7,Tareq,AbuAli,customer7@example.com,Amman,Negotiation
8,Tareq,Shami,<NA>,Aqaba,Qualified
9,Yousef,Hamdan,customer9@example.com,Unknown,Won


## Step 5 — Summary

**Done**
- Stripped whitespace on text columns
- Standardized `Region`, `Customer_Segment`, `Lead_Source`, `Lead_Stage`, `Customer_Status`
- Title-cased `First_Name` / `Last_Name` (merged dup spellings like `ADAM` / `Adam`)
- Lowercased `Email` where present

**Working frame:** `df_clean`

---

Step 5 complete. Continue below for **Step 6 — Outliers + invalid / logical checks**.

## Step 6 — Outliers + invalid / logical checks

**Why this step?** Invalid numbers quietly break KPIs (average satisfaction > 10, negative revenue, negative response time).

We separate:
- **Invalid values** → must fix (out of allowed domain)
- **Statistical outliers** → review / flag, usually **keep** (real large deals exist)
- **Business logic** → document; only fix when clearly impossible

### 6.1 Numeric profile + domain violations

In [26]:
numeric_cols = [
    "Lead_Score",
    "Interaction_Count",
    "Website_Visits",
    "Email_Opens",
    "Response_Time_Hours",
    "Customer_Satisfaction",
    "Revenue",
]

df_clean[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Lead_Score,4875.0,59.407385,23.170254,1.0,43.0,60.0,76.00,120.00
Interaction_Count,5000.0,5.015400,2.259775,0.0,3.0,5.0,6.00,14.00
Website_Visits,5000.0,7.985400,2.825524,0.0,6.0,8.0,10.00,21.00
Email_Opens,5000.0,3.991800,2.003231,0.0,3.0,4.0,5.00,13.00
Response_Time_Hours,4825.0,11.162922,7.466550,-5.0,5.7,9.7,15.00,61.60
Customer_Satisfaction,4726.0,7.378142,1.492471,2.2,6.4,7.4,8.40,15.00
Revenue,5000.0,327.939314,1242.480892,-500.0,0.0,0.0,154.47,23346.24


In [27]:
invalid_checks = pd.DataFrame([
    {
        "rule": "Lead_Score outside 0–100",
        "count": int(((df_clean["Lead_Score"] < 0) | (df_clean["Lead_Score"] > 100)).sum()),
    },
    {
        "rule": "Customer_Satisfaction outside 0–10",
        "count": int(((df_clean["Customer_Satisfaction"] < 0) | (df_clean["Customer_Satisfaction"] > 10)).sum()),
    },
    {
        "rule": "Response_Time_Hours < 0",
        "count": int((df_clean["Response_Time_Hours"] < 0).sum()),
    },
    {
        "rule": "Revenue < 0",
        "count": int((df_clean["Revenue"] < 0).sum()),
    },
    {
        "rule": "Interaction_Count < 0",
        "count": int((df_clean["Interaction_Count"] < 0).sum()),
    },
    {
        "rule": "Website_Visits < 0",
        "count": int((df_clean["Website_Visits"] < 0).sum()),
    },
    {
        "rule": "Email_Opens < 0",
        "count": int((df_clean["Email_Opens"] < 0).sum()),
    },
])
invalid_checks

,rule,count
0,Lead_Score outside 0–100,10
1,Customer_Satisfaction outside 0–10,8
2,Response_Time_Hours < 0,9
3,Revenue < 0,7
4,Interaction_Count < 0,0
5,Website_Visits < 0,0
6,Email_Opens < 0,0


### 6.2 Handling rules (invalid vs outliers)

| Issue | Action | Why |
|---|---|---|
| `Lead_Score` > 100 | **Cap at 100** | Score scale is 0–100; values like 120 are domain errors |
| `Customer_Satisfaction` > 10 | **Cap at 10** | Typical 0–10 CSAT scale |
| `Response_Time_Hours` < 0 | **Set to NaN** | Time cannot be negative (we already dropped one `-5` near-dupe earlier; others remain) |
| `Revenue` < 0 | **Set to NaN** | Negative closed revenue is not valid here (`-500` looks like a bad sentinel) |
| High `Revenue` / high `Response_Time_Hours` (IQR) | **Keep + report** | Large won deals and slow responses can be real |
| `Lead_Stage` in Proposal/Negotiation with `Revenue` > 0 | **Keep** | Treat as pipeline / expected value, not necessarily won revenue |
| `Won` with `Revenue` == 0 | Check only | In this dataset: should be 0 after cleaning |

In [28]:
# Capture before counts for the fix report
before = {
    "lead_score_gt_100": int((df_clean["Lead_Score"] > 100).sum()),
    "csat_gt_10": int((df_clean["Customer_Satisfaction"] > 10).sum()),
    "rt_neg": int((df_clean["Response_Time_Hours"] < 0).sum()),
    "revenue_neg": int((df_clean["Revenue"] < 0).sum()),
}

# Apply fixes
df_clean["Lead_Score"] = df_clean["Lead_Score"].clip(upper=100)
df_clean["Customer_Satisfaction"] = df_clean["Customer_Satisfaction"].clip(upper=10)

rt_neg_mask = df_clean["Response_Time_Hours"] < 0
df_clean.loc[rt_neg_mask, "Response_Time_Hours"] = np.nan

rev_neg_mask = df_clean["Revenue"] < 0
df_clean.loc[rev_neg_mask, "Revenue"] = np.nan

fix_report = pd.DataFrame([
    {"fix": "Lead_Score capped at 100", "rows_affected": before["lead_score_gt_100"]},
    {"fix": "Customer_Satisfaction capped at 10", "rows_affected": before["csat_gt_10"]},
    {"fix": "Response_Time_Hours < 0 → NaN", "rows_affected": before["rt_neg"]},
    {"fix": "Revenue < 0 → NaN", "rows_affected": before["revenue_neg"]},
])
fix_report

,fix,rows_affected
0,Lead_Score capped at 100,10
1,Customer_Satisfaction capped at 10,8
2,Response_Time_Hours < 0 → NaN,9
3,Revenue < 0 → NaN,7


### 6.3 Re-check domain rules + logical audit (no row drops)

In [29]:
print("Remaining domain violations (should be 0):")
print("  Lead_Score > 100:", int((df_clean["Lead_Score"] > 100).sum()))
print("  CSAT > 10:", int((df_clean["Customer_Satisfaction"] > 10).sum()))
print("  Response_Time < 0:", int((df_clean["Response_Time_Hours"] < 0).sum()))
print("  Revenue < 0:", int((df_clean["Revenue"] < 0).sum()))

# Logical / business observations
won = df_clean["Lead_Stage"] == "Won"
logical_audit = pd.DataFrame([
    {
        "check": "Won with Revenue == 0",
        "count": int((won & (df_clean["Revenue"].fillna(0) == 0)).sum()),
    },
    {
        "check": "Won with Revenue > 0",
        "count": int((won & (df_clean["Revenue"] > 0)).sum()),
    },
    {
        "check": "Not Won but Revenue > 0 (pipeline value — kept)",
        "count": int((~won & (df_clean["Revenue"] > 0)).sum()),
    },
    {
        "check": "Email_Opens > Interaction_Count (possible; kept)",
        "count": int((df_clean["Email_Opens"] > df_clean["Interaction_Count"]).sum()),
    },
])
logical_audit

Remaining domain violations (should be 0):
  Lead_Score > 100: 0
  CSAT > 10: 0
  Response_Time < 0: 0
  Revenue < 0: 0


,check,count
0,Won with Revenue == 0,0
1,Won with Revenue > 0,819
2,Not Won but Revenue > 0 (pipeline va...,1211
3,Email_Opens > Interaction_Count (pos...,1515


### 6.4 Statistical outliers (report only — keep rows)

IQR fence on `Response_Time_Hours` and `Revenue` (among positive revenue). We do **not** delete these; they are candidates for dashboard callouts.

In [30]:
def iqr_outlier_count(series: pd.Series) -> dict:
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {
        "q1": round(float(q1), 2),
        "q3": round(float(q3), 2),
        "low_fence": round(float(low), 2),
        "high_fence": round(float(high), 2),
        "n_below": int((s < low).sum()),
        "n_above": int((s > high).sum()),
    }

outlier_report = pd.DataFrame({
    "Response_Time_Hours": iqr_outlier_count(df_clean["Response_Time_Hours"]),
    "Revenue_positive": iqr_outlier_count(df_clean.loc[df_clean["Revenue"] > 0, "Revenue"]),
}).T
outlier_report

,q1,q3,low_fence,high_fence,n_below,n_above
Response_Time_Hours,5.70,15.00,-8.25,28.95,0.0,136.0
Revenue_positive,92.67,743.94,-884.23,1720.84,0.0,222.0


## Step 6 — Summary

**Fixed (invalid domain)**
- Capped `Lead_Score` at 100 and `Customer_Satisfaction` at 10
- Set negative `Response_Time_Hours` and negative `Revenue` to NaN

**Kept (documented)**
- High IQR outliers on response time / revenue
- Non-won stages with revenue > 0 (pipeline value)
- `Email_Opens` > `Interaction_Count`

**Working frame:** `df_clean`

---

Step 6 complete. Continue below for **Step 7 — Final transforms + export**.

## Step 7 — Final transforms + export

**Why this step?** Lock a clean, analysis-ready table:
- Add a few derived fields useful for EDA / dashboard KPIs
- Reorder columns for readability
- Final QA snapshot
- Export `CRM_Cleaned.csv`

### 7.1 Derived columns

| New column | Logic |
|---|---|
| `Full_Name` | `First_Name` + `Last_Name` |
| `Days_to_Last_Contact` | `Last_Contact_Date - Signup_Date` in days (null if no last contact) |
| `Is_Won` | `Lead_Stage == "Won"` |
| `Has_Revenue` | `Revenue > 0` |

In [31]:
df_clean["Full_Name"] = (
    df_clean["First_Name"].astype("string") + " " + df_clean["Last_Name"].astype("string")
)

df_clean["Days_to_Last_Contact"] = (
    df_clean["Last_Contact_Date"] - df_clean["Signup_Date"]
).dt.days

df_clean["Is_Won"] = df_clean["Lead_Stage"].eq("Won")
df_clean["Has_Revenue"] = df_clean["Revenue"].fillna(0).gt(0)

print("Derived columns added:")
df_clean[["Full_Name", "Days_to_Last_Contact", "Is_Won", "Has_Revenue"]].head()

Derived columns added:


,Full_Name,Days_to_Last_Contact,Is_Won,Has_Revenue
0,Noor Hassan,49.0,False,False
1,Dana Odeh,125.0,False,False
2,Yara Khalil,13.0,False,True
3,Yara Saleh,36.0,False,False
4,Zaid Hamdan,101.0,False,False


### 7.2 Column order + final QA

In [32]:
final_column_order = [
    "Customer_ID",
    "Full_Name",
    "First_Name",
    "Last_Name",
    "Email",
    "Phone",
    "Region",
    "Customer_Segment",
    "Industry",
    "Lead_Source",
    "Lead_Stage",
    "Is_Won",
    "Customer_Status",
    "Product",
    "Campaign",
    "Preferred_Channel",
    "Assigned_Sales_Rep",
    "Signup_Date",
    "Last_Contact_Date",
    "Days_to_Last_Contact",
    "Lead_Score",
    "Interaction_Count",
    "Website_Visits",
    "Email_Opens",
    "Response_Time_Hours",
    "Customer_Satisfaction",
    "Revenue",
    "Has_Revenue",
]

df_clean = df_clean[final_column_order].sort_values("Customer_ID").reset_index(drop=True)

qa = pd.DataFrame({
    "metric": [
        "rows",
        "columns",
        "unique Customer_ID",
        "duplicate Customer_ID",
        "duplicate full rows",
        "Won leads",
        "rows with Revenue > 0",
        "missing Email",
        "missing Phone",
        "missing Lead_Score",
        "missing CSAT",
        "missing Response_Time",
        "missing Last_Contact_Date",
    ],
    "value": [
        len(df_clean),
        df_clean.shape[1],
        df_clean["Customer_ID"].nunique(),
        int(df_clean["Customer_ID"].duplicated().sum()),
        int(df_clean.duplicated().sum()),
        int(df_clean["Is_Won"].sum()),
        int(df_clean["Has_Revenue"].sum()),
        int(df_clean["Email"].isna().sum()),
        int(df_clean["Phone"].isna().sum()),
        int(df_clean["Lead_Score"].isna().sum()),
        int(df_clean["Customer_Satisfaction"].isna().sum()),
        int(df_clean["Response_Time_Hours"].isna().sum()),
        int(df_clean["Last_Contact_Date"].isna().sum()),
    ],
})
qa

,metric,value
0,rows,5000
1,columns,28
2,unique Customer_ID,5000
3,duplicate Customer_ID,0
4,duplicate full rows,0
5,Won leads,819
6,rows with Revenue > 0,2030
7,missing Email,125
8,missing Phone,200
9,missing Lead_Score,125


### 7.3 Export `CRM_Cleaned.csv`

Saved next to the raw file for Phase 2 EDA / dashboard.

In [33]:
output_path = "CRM_Cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"Exported: {output_path}")
print(f"Shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
df_clean.head()

Exported: CRM_Cleaned.csv
Shape: 5,000 rows × 28 columns


,Customer_ID,Full_Name,First_Name,Last_Name,Email,Phone,Region,Customer_Segment,Industry,Lead_Source,Lead_Stage,Is_Won,Customer_Status,Product,Campaign,Preferred_Channel,Assigned_Sales_Rep,Signup_Date,Last_Contact_Date,Days_to_Last_Contact,Lead_Score,Interaction_Count,Website_Visits,Email_Opens,Response_Time_Hours,Customer_Satisfaction,Revenue,Has_Revenue
0,CUST10000,Noor Hassan,Noor,Hassan,customer0@example.com,962716349958,Aqaba,Corporate,Technology,LinkedIn,New Lead,False,Active,CRM Enterprise,No Campaign,Social Media,Lina Haddad,2025-01-30,2025-03-20,49.0,46.0,11,8,7,8.8,7.5,0.00,False
1,CUST10001,Dana Odeh,Dana,Odeh,customer1@example.com,962753765818,Aqaba,Corporate,Construction,Email Campaign,Contacted,False,Active,Marketing Add-on,Referral Program,Meeting,Omar Mansour,2024-10-18,2025-02-20,125.0,76.0,5,6,3,14.6,9.5,0.00,False
2,CUST10002,Yara Khalil,Yara,Khalil,customer2@example.com,962784357962,Salt,Consumer,Healthcare,Unknown,Proposal,False,Inactive,CRM Professional,No Campaign,Phone,Omar Mansour,2024-10-20,2024-11-02,13.0,88.0,7,11,1,8.8,7.1,69.84,True
3,CUST10003,Yara Saleh,Yara,Saleh,customer3@example.com,962784969084,Karak,Consumer,Technology,Google Ads,Qualified,False,Active,Marketing Add-on,Back-to-School,WhatsApp,Yara Odeh,2023-09-02,2023-10-08,36.0,28.0,4,9,9,7.1,10.0,0.00,False
4,CUST10004,Zaid Hamdan,Zaid,Hamdan,customer4@example.com,962796712050,Salt,Corporate,Retail,Instagram,Qualified,False,Prospect,CRM Basic,Summer Campaign,Social Media,Lina Haddad,2023-06-04,2023-09-13,101.0,100.0,7,8,2,7.1,9.4,0.00,False


## Step 7 — Summary / Phase 1 complete

**Exported:** `CRM_Cleaned.csv`

**Phase 1 cleaning pipeline**
1. Inspected raw CRM extract
2. Filled key category nulls with `Unknown`; left contacts/scores/dates as NaN
3. Removed exact duplicates + resolved 1 conflicting `Customer_ID`
4. Parsed dates, swapped 20 inverted pairs, fixed Phone dtype
5. Standardized text categories and names
6. Fixed invalid scores/times/revenue; kept statistical outliers
7. Added derived fields and exported cleaned file

**Next (when you say so):** Phase 2 — EDA notebook on `CRM_Cleaned.csv` (then Matplotlib / Seaborn / Plotly dashboard).

In [2]:
import pandas as pd
import numpy as np

# Load cleaned CRM dataset
df_clean = pd.read_csv("CRM_Cleaned.csv")

print("Dataset loaded successfully.")
print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns: {df_clean.shape[1]}")

Dataset loaded successfully.
Rows: 5,000
Columns: 28


In [3]:
print("=" * 60)
print("FINAL CRM DATA QUALITY CHECK")
print("=" * 60)

# 1. Basic information
print("\n[1] BASIC DATASET CHECK")
print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns: {df_clean.shape[1]}")

# 2. Duplicate rows
print("\n[2] DUPLICATE CHECK")
print(f"Duplicate rows: {df_clean.duplicated().sum():,}")

# 3. Customer ID duplicates
print("\n[3] CUSTOMER ID CHECK")
print(f"Duplicate Customer_ID: {df_clean['Customer_ID'].duplicated().sum():,}")

# 4. Missing values
print("\n[4] MISSING VALUES CHECK")

missing = df_clean.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) == 0:
    print("PASS - No missing values")
else:
    print("Missing values:")
    print(missing)

# 5. Data types
print("\n[5] DATA TYPES")
print(df_clean.dtypes)

# 6. Numeric validation
print("\n[6] NUMERIC VALIDATION")

print(
    "Lead_Score outside 0-100:",
    ((df_clean["Lead_Score"] < 0) | 
     (df_clean["Lead_Score"] > 100)).sum()
)

print(
    "Customer_Satisfaction outside 0-10:",
    ((df_clean["Customer_Satisfaction"] < 0) | 
     (df_clean["Customer_Satisfaction"] > 10)).sum()
)

print(
    "Negative Response_Time:",
    (df_clean["Response_Time_Hours"] < 0).sum()
)

print(
    "Negative Revenue:",
    (df_clean["Revenue"] < 0).sum()
)

# 7. Date validation
print("\n[7] DATE VALIDATION")

print("Signup_Date dtype:", df_clean["Signup_Date"].dtype)
print("Last_Contact_Date dtype:", df_clean["Last_Contact_Date"].dtype)

# Convert temporarily for checking
signup = pd.to_datetime(df_clean["Signup_Date"], errors="coerce")
last_contact = pd.to_datetime(df_clean["Last_Contact_Date"], errors="coerce")

invalid_dates = (
    last_contact < signup
).sum()

print(
    "Last_Contact_Date before Signup_Date:",
    invalid_dates
)

# 8. Lead Stage
print("\n[8] LEAD STAGE")

print(
    df_clean["Lead_Stage"]
    .value_counts(dropna=False)
)

# 9. Revenue by Lead Stage
print("\n[9] REVENUE BY LEAD STAGE")

print(
    df_clean.groupby("Lead_Stage")["Revenue"]
    .agg(["count", "sum", "mean"])
    .round(2)
)

# 10. Final summary
print("\n" + "=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print(f"Rows: {len(df_clean):,}")
print(f"Columns: {len(df_clean.columns)}")
print(f"Duplicate rows: {df_clean.duplicated().sum():,}")
print(f"Missing cells: {df_clean.isna().sum().sum():,}")

print("\nCHECK COMPLETED")

FINAL CRM DATA QUALITY CHECK

[1] BASIC DATASET CHECK
Rows: 5,000
Columns: 28

[2] DUPLICATE CHECK
Duplicate rows: 0

[3] CUSTOMER ID CHECK
Duplicate Customer_ID: 0

[4] MISSING VALUES CHECK
Missing values:
Customer_Satisfaction    274
Days_to_Last_Contact     224
Last_Contact_Date        224
Phone                    200
Response_Time_Hours      184
Email                    125
Lead_Score               125
Revenue                    7
dtype: int64

[5] DATA TYPES
Customer_ID                  str
Full_Name                    str
First_Name                   str
Last_Name                    str
Email                        str
Phone                    float64
Region                       str
Customer_Segment             str
Industry                     str
Lead_Source                  str
Lead_Stage                   str
Is_Won                      bool
Customer_Status              str
Product                      str
Campaign                     str
Preferred_Channel            str
Assi

In [4]:
print("=== FINAL CHECK ===")

print("\n1. Lead Score invalid:")
print(((df_clean["Lead_Score"] < 0) | 
       (df_clean["Lead_Score"] > 100)).sum())

print("\n2. Customer Satisfaction invalid:")
print(((df_clean["Customer_Satisfaction"] < 0) | 
       (df_clean["Customer_Satisfaction"] > 10)).sum())

print("\n3. Negative Response Time:")
print((df_clean["Response_Time_Hours"] < 0).sum())

print("\n4. Negative Revenue:")
print((df_clean["Revenue"] < 0).sum())

print("\n5. Invalid Dates:")

signup = pd.to_datetime(df_clean["Signup_Date"], errors="coerce")
last_contact = pd.to_datetime(
    df_clean["Last_Contact_Date"],
    errors="coerce"
)

print((last_contact < signup).sum())

print("\n6. Lead Stages:")
print(df_clean["Lead_Stage"].value_counts(dropna=False))

print("\n7. Revenue by Lead Stage:")
print(
    df_clean.groupby("Lead_Stage")["Revenue"]
    .agg(["count", "sum", "mean"])
    .round(2)
)

print("\n=== END ===")

=== FINAL CHECK ===

1. Lead Score invalid:
0

2. Customer Satisfaction invalid:
0

3. Negative Response Time:
0

4. Negative Revenue:
0

5. Invalid Dates:
0

6. Lead Stages:
Lead_Stage
Qualified      893
Contacted      882
Won            819
New Lead       761
Proposal       689
Negotiation    523
Lost           433
Name: count, dtype: int64

7. Revenue by Lead Stage:
             count         sum     mean
Lead_Stage                             
Contacted      881        0.00     0.00
Lost           432        0.00     0.00
Negotiation    522   292319.00   560.00
New Lead       760        0.00     0.00
Proposal       689   164282.81   238.44
Qualified      890        0.00     0.00
Won            819  1186594.76  1448.83

=== END ===


In [5]:
# ============================================
# CRM BUSINESS ANALYSIS - OVERVIEW
# ============================================

print("=" * 60)
print("CRM BUSINESS ANALYSIS")
print("=" * 60)

# 1. Basic KPIs
total_customers = df_clean["Customer_ID"].nunique()
total_revenue = df_clean["Revenue"].sum()
total_leads = len(df_clean)
won_leads = (df_clean["Lead_Stage"] == "Won").sum()

conversion_rate = won_leads / total_leads * 100

print("\n--- KEY BUSINESS KPIs ---")
print(f"Total Customers: {total_customers:,}")
print(f"Total Leads: {total_leads:,}")
print(f"Won Leads: {won_leads:,}")
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Conversion Rate: {conversion_rate:.2f}%")


# 2. Lead Funnel
print("\n--- LEAD FUNNEL ---")

funnel = (
    df_clean["Lead_Stage"]
    .value_counts()
    .reindex([
        "New Lead",
        "Contacted",
        "Qualified",
        "Proposal",
        "Negotiation",
        "Won",
        "Lost"
    ])
    .fillna(0)
    .astype(int)
)

print(funnel)


# 3. Revenue by Customer Segment
print("\n--- REVENUE BY CUSTOMER SEGMENT ---")

segment_analysis = (
    df_clean.groupby("Customer_Segment")
    .agg(
        Leads=("Customer_ID", "count"),
        Revenue=("Revenue", "sum"),
        Average_Revenue=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

print(segment_analysis.round(2))


# 4. Revenue by Region
print("\n--- REVENUE BY REGION ---")

region_analysis = (
    df_clean.groupby("Region")
    .agg(
        Leads=("Customer_ID", "count"),
        Revenue=("Revenue", "sum"),
        Average_Revenue=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

print(region_analysis.round(2))


# 5. Revenue by Industry
print("\n--- REVENUE BY INDUSTRY ---")

industry_analysis = (
    df_clean.groupby("Industry")
    .agg(
        Leads=("Customer_ID", "count"),
        Revenue=("Revenue", "sum"),
        Average_Revenue=("Revenue", "mean")
    )
    .sort_values("Revenue", ascending=False)
)

print(industry_analysis.round(2))


print("\n" + "=" * 60)
print("ANALYSIS COMPLETED")
print("=" * 60)

CRM BUSINESS ANALYSIS

--- KEY BUSINESS KPIs ---
Total Customers: 5,000
Total Leads: 5,000
Won Leads: 819
Total Revenue: $1,643,196.57
Conversion Rate: 16.38%

--- LEAD FUNNEL ---
Lead_Stage
New Lead       761
Contacted      882
Qualified      893
Proposal       689
Negotiation    523
Won            819
Lost           433
Name: count, dtype: int64

--- REVENUE BY CUSTOMER SEGMENT ---
                  Leads    Revenue  Average_Revenue
Customer_Segment                                   
Enterprise          487  639056.44          1317.64
Corporate          1040  578766.61           558.12
Small Business     1091  242106.61           221.91
Consumer           2382  183266.91            77.00

--- REVENUE BY REGION ---
         Leads    Revenue  Average_Revenue
Region                                    
Irbid      646  249681.17           387.10
Amman      660  233262.73           353.43
Jerash     642  207815.03           324.20
Madaba     596  207057.90           348.00
Aqaba      603  

In [6]:
print("\n--- REVENUE BY CUSTOMER SEGMENT ---")

print(
    segment_analysis
    .round(2)
    .to_string()
)


print("\n--- REVENUE BY REGION ---")

print(
    region_analysis
    .round(2)
    .to_string()
)


print("\n--- REVENUE BY INDUSTRY ---")

print(
    industry_analysis
    .round(2)
    .to_string()
)


--- REVENUE BY CUSTOMER SEGMENT ---
                  Leads    Revenue  Average_Revenue
Customer_Segment                                   
Enterprise          487  639056.44          1317.64
Corporate          1040  578766.61           558.12
Small Business     1091  242106.61           221.91
Consumer           2382  183266.91            77.00

--- REVENUE BY REGION ---
         Leads    Revenue  Average_Revenue
Region                                    
Irbid      646  249681.17           387.10
Amman      660  233262.73           353.43
Jerash     642  207815.03           324.20
Madaba     596  207057.90           348.00
Aqaba      603  201185.88           334.75
Salt       599  194152.82           324.13
Karak      619  159311.71           258.20
Zarqa      545  153266.27           281.22
Unknown     90   37463.06           416.26

--- REVENUE BY INDUSTRY ---
               Leads    Revenue  Average_Revenue
Industry                                        
Finance          599  23

In [7]:
# ============================================
# CONVERSION RATE ANALYSIS
# ============================================

print("=" * 60)
print("CONVERSION RATE ANALYSIS")
print("=" * 60)


# Overall conversion
overall_conversion = (
    (df_clean["Lead_Stage"] == "Won").sum()
    / len(df_clean)
    * 100
)

print(f"\nOverall Conversion Rate: {overall_conversion:.2f}%")


# Conversion by Customer Segment
segment_conversion = (
    df_clean.groupby("Customer_Segment")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum")
    )
)

segment_conversion["Conversion_Rate_%"] = (
    segment_conversion["Won_Leads"]
    / segment_conversion["Leads"]
    * 100
)

segment_conversion = segment_conversion.sort_values(
    "Conversion_Rate_%",
    ascending=False
)

print("\n--- CONVERSION BY CUSTOMER SEGMENT ---")
print(segment_conversion.round(2).to_string())


# Conversion by Region
region_conversion = (
    df_clean.groupby("Region")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum")
    )
)

region_conversion["Conversion_Rate_%"] = (
    region_conversion["Won_Leads"]
    / region_conversion["Leads"]
    * 100
)

region_conversion = region_conversion.sort_values(
    "Conversion_Rate_%",
    ascending=False
)

print("\n--- CONVERSION BY REGION ---")
print(region_conversion.round(2).to_string())


# Conversion by Industry
industry_conversion = (
    df_clean.groupby("Industry")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum")
    )
)

industry_conversion["Conversion_Rate_%"] = (
    industry_conversion["Won_Leads"]
    / industry_conversion["Leads"]
    * 100
)

industry_conversion = industry_conversion.sort_values(
    "Conversion_Rate_%",
    ascending=False
)

print("\n--- CONVERSION BY INDUSTRY ---")
print(industry_conversion.round(2).to_string())

CONVERSION RATE ANALYSIS

Overall Conversion Rate: 16.38%

--- CONVERSION BY CUSTOMER SEGMENT ---
                  Leads  Won_Leads  Conversion_Rate_%
Customer_Segment                                     
Small Business     1091        182              16.68
Consumer           2382        395              16.58
Enterprise          487         80              16.43
Corporate          1040        162              15.58

--- CONVERSION BY REGION ---
         Leads  Won_Leads  Conversion_Rate_%
Region                                      
Jerash     642        123              19.16
Amman      660        116              17.58
Irbid      646        110              17.03
Karak      619        100              16.16
Aqaba      603         96              15.92
Salt       599         92              15.36
Zarqa      545         83              15.23
Madaba     596         90              15.10
Unknown     90          9              10.00

--- CONVERSION BY INDUSTRY ---
               Leads 

In [8]:
# ============================================
# LEAD SOURCE PERFORMANCE ANALYSIS
# ============================================

print("=" * 60)
print("LEAD SOURCE PERFORMANCE")
print("=" * 60)

lead_source_analysis = (
    df_clean.groupby("Lead_Source")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum"),
        Average_Revenue=("Revenue", "mean")
    )
)

lead_source_analysis["Conversion_Rate_%"] = (
    lead_source_analysis["Won_Leads"]
    / lead_source_analysis["Leads"]
    * 100
)

lead_source_analysis = lead_source_analysis[
    [
        "Leads",
        "Won_Leads",
        "Conversion_Rate_%",
        "Revenue",
        "Average_Revenue"
    ]
].sort_values(
    "Conversion_Rate_%",
    ascending=False
)

print(
    lead_source_analysis
    .round(2)
    .to_string()
)

LEAD SOURCE PERFORMANCE
                Leads  Won_Leads  Conversion_Rate_%    Revenue  Average_Revenue
Lead_Source                                                                    
Website           562        104              18.51  225465.69           401.90
LinkedIn          552        101              18.30  217639.37           394.99
Facebook          553         93              16.82  179357.27           324.34
Cold Call         537         90              16.76  148761.23           277.02
Trade Show        533         88              16.51  209310.11           393.44
Email Campaign    569         93              16.34  210565.59           370.06
Unknown           150         24              16.00   33753.25           225.02
Referral          490         77              15.71  110577.55           226.13
Google Ads        520         78              15.00  157746.49           303.94
Instagram         534         71              13.30  150020.02           281.99


In [9]:
# ============================================
# CAMPAIGN PERFORMANCE ANALYSIS
# ============================================

print("=" * 60)
print("CAMPAIGN PERFORMANCE")
print("=" * 60)

campaign_analysis = (
    df_clean.groupby("Campaign")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum"),
        Average_Revenue=("Revenue", "mean")
    )
)

campaign_analysis["Conversion_Rate_%"] = (
    campaign_analysis["Won_Leads"]
    / campaign_analysis["Leads"]
    * 100
)

campaign_analysis = campaign_analysis[
    [
        "Leads",
        "Won_Leads",
        "Conversion_Rate_%",
        "Revenue",
        "Average_Revenue"
    ]
].sort_values(
    "Conversion_Rate_%",
    ascending=False
)

print(
    campaign_analysis
    .round(2)
    .to_string()
)

CAMPAIGN PERFORMANCE
                   Leads  Won_Leads  Conversion_Rate_%    Revenue  Average_Revenue
Campaign                                                                          
Q1 Acquisition       773        140              18.11  324448.43           420.27
Year-End Campaign    652        116              17.79  224049.59           344.69
Spring Campaign      632        111              17.56  231784.30           367.33
No Campaign          930        152              16.34  281375.99           302.88
Referral Program     584         90              15.41  182347.66           312.24
Unknown              150         23              15.33   48909.33           326.06
Summer Campaign      644         98              15.22  170175.08           265.07
Back-to-School       635         89              14.02  180106.19           283.63


In [10]:
# ============================================
# SALES REPRESENTATIVE PERFORMANCE
# ============================================

print("=" * 60)
print("SALES REPRESENTATIVE PERFORMANCE")
print("=" * 60)

sales_rep_analysis = (
    df_clean.groupby("Assigned_Sales_Rep")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum"),
        Average_Revenue=("Revenue", "mean")
    )
)

sales_rep_analysis["Conversion_Rate_%"] = (
    sales_rep_analysis["Won_Leads"]
    / sales_rep_analysis["Leads"]
    * 100
)

sales_rep_analysis = sales_rep_analysis[
    [
        "Leads",
        "Won_Leads",
        "Conversion_Rate_%",
        "Revenue",
        "Average_Revenue"
    ]
].sort_values(
    "Revenue",
    ascending=False
)

print(
    sales_rep_analysis
    .round(2)
    .to_string()
)

SALES REPRESENTATIVE PERFORMANCE
                    Leads  Won_Leads  Conversion_Rate_%    Revenue  Average_Revenue
Assigned_Sales_Rep                                                                 
Samer Khalil          848        144              16.98  329110.48           388.56
Yara Odeh             850        121              14.24  319138.01           375.90
Omar Mansour          811        141              17.39  270459.56           335.14
Lina Haddad           808        134              16.58  259539.06           321.21
Ahmad Saleh           816        152              18.63  258593.30           317.29
Dana Nasser           867        127              14.65  206356.16           238.01


In [11]:
# ============================================
# PRODUCT PERFORMANCE ANALYSIS
# ============================================

print("=" * 60)
print("PRODUCT PERFORMANCE")
print("=" * 60)

product_analysis = (
    df_clean.groupby("Product")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum"),
        Average_Revenue=("Revenue", "mean")
    )
)

product_analysis["Conversion_Rate_%"] = (
    product_analysis["Won_Leads"]
    / product_analysis["Leads"]
    * 100
)

product_analysis = product_analysis[
    [
        "Leads",
        "Won_Leads",
        "Conversion_Rate_%",
        "Revenue",
        "Average_Revenue"
    ]
].sort_values(
    "Revenue",
    ascending=False
)

print(
    product_analysis
    .round(2)
    .to_string()
)

PRODUCT PERFORMANCE
                  Leads  Won_Leads  Conversion_Rate_%    Revenue  Average_Revenue
Product                                                                          
CRM Enterprise      907        133              14.66  755005.62           833.34
CRM Professional   1491        242              16.23  417576.33           280.82
CRM Basic          1414        245              17.33  205708.18           145.69
Analytics Add-on    582        105              18.04  156067.17           268.16
Marketing Add-on    606         94              15.51  108839.27           179.60


In [12]:
# ============================================
# CUSTOMER EXPERIENCE ANALYSIS
# ============================================

print("=" * 60)
print("CUSTOMER EXPERIENCE ANALYSIS")
print("=" * 60)


# 1. Overall Customer Experience
print("\n--- OVERALL CUSTOMER EXPERIENCE ---")

avg_response_time = df_clean["Response_Time_Hours"].mean()
avg_satisfaction = df_clean["Customer_Satisfaction"].mean()

print(f"Average Response Time: {avg_response_time:.2f} hours")
print(f"Average Customer Satisfaction: {avg_satisfaction:.2f} / 10")


# 2. Response Time by Customer Status
print("\n--- RESPONSE TIME BY CUSTOMER STATUS ---")

response_by_status = (
    df_clean.groupby("Customer_Status")
    .agg(
        Customers=("Customer_ID", "count"),
        Avg_Response_Time=("Response_Time_Hours", "mean"),
        Avg_Satisfaction=("Customer_Satisfaction", "mean")
    )
    .sort_values("Avg_Response_Time")
)

print(
    response_by_status
    .round(2)
    .to_string()
)


# 3. Customer Satisfaction by Customer Status
print("\n--- SATISFACTION BY CUSTOMER STATUS ---")

satisfaction_by_status = (
    df_clean.groupby("Customer_Status")
    .agg(
        Customers=("Customer_ID", "count"),
        Avg_Satisfaction=("Customer_Satisfaction", "mean"),
        Avg_Response_Time=("Response_Time_Hours", "mean")
    )
    .sort_values("Avg_Satisfaction", ascending=False)
)

print(
    satisfaction_by_status
    .round(2)
    .to_string()
)


# 4. Response Time by Lead Stage
print("\n--- RESPONSE TIME BY LEAD STAGE ---")

response_by_stage = (
    df_clean.groupby("Lead_Stage")
    .agg(
        Leads=("Customer_ID", "count"),
        Avg_Response_Time=("Response_Time_Hours", "mean"),
        Avg_Satisfaction=("Customer_Satisfaction", "mean")
    )
    .sort_values("Avg_Response_Time")
)

print(
    response_by_stage
    .round(2)
    .to_string()
)

CUSTOMER EXPERIENCE ANALYSIS

--- OVERALL CUSTOMER EXPERIENCE ---
Average Response Time: 11.19 hours
Average Customer Satisfaction: 7.37 / 10

--- RESPONSE TIME BY CUSTOMER STATUS ---
                 Customers  Avg_Response_Time  Avg_Satisfaction
Customer_Status                                                
Inactive               904              11.02              7.36
Prospect               936              11.07              7.38
Active                3160              11.28              7.37

--- SATISFACTION BY CUSTOMER STATUS ---
                 Customers  Avg_Satisfaction  Avg_Response_Time
Customer_Status                                                
Prospect               936              7.38              11.07
Active                3160              7.37              11.28
Inactive               904              7.36              11.02

--- RESPONSE TIME BY LEAD STAGE ---
             Leads  Avg_Response_Time  Avg_Satisfaction
Lead_Stage                                

In [13]:
# ============================================
# LEAD QUALITY & ENGAGEMENT ANALYSIS
# ============================================

print("=" * 60)
print("LEAD QUALITY & ENGAGEMENT ANALYSIS")
print("=" * 60)


# 1. Overall Lead Quality & Engagement
print("\n--- OVERALL METRICS ---")

print(
    f"Average Lead Score: "
    f"{df_clean['Lead_Score'].mean():.2f}"
)

print(
    f"Average Interactions: "
    f"{df_clean['Interaction_Count'].mean():.2f}"
)

print(
    f"Average Website Visits: "
    f"{df_clean['Website_Visits'].mean():.2f}"
)

print(
    f"Average Email Opens: "
    f"{df_clean['Email_Opens'].mean():.2f}"
)


# 2. Won vs Not Won
print("\n--- WON VS NOT WON ---")

won_analysis = (
    df_clean.groupby("Is_Won")
    .agg(
        Leads=("Customer_ID", "count"),
        Avg_Lead_Score=("Lead_Score", "mean"),
        Avg_Interactions=("Interaction_Count", "mean"),
        Avg_Website_Visits=("Website_Visits", "mean"),
        Avg_Email_Opens=("Email_Opens", "mean"),
        Avg_Satisfaction=("Customer_Satisfaction", "mean")
    )
)

print(
    won_analysis
    .round(2)
    .to_string()
)


# 3. Lead Score by Lead Stage
print("\n--- LEAD SCORE BY STAGE ---")

score_by_stage = (
    df_clean.groupby("Lead_Stage")
    .agg(
        Leads=("Customer_ID", "count"),
        Avg_Lead_Score=("Lead_Score", "mean"),
        Avg_Interactions=("Interaction_Count", "mean"),
        Avg_Website_Visits=("Website_Visits", "mean"),
        Avg_Email_Opens=("Email_Opens", "mean")
    )
    .sort_values(
        "Avg_Lead_Score",
        ascending=False
    )
)

print(
    score_by_stage
    .round(2)
    .to_string()
)


# 4. Lead Score Buckets
print("\n--- LEAD SCORE BUCKET ANALYSIS ---")

df_clean["Lead_Score_Band"] = pd.cut(
    df_clean["Lead_Score"],
    bins=[0, 40, 60, 80, 100],
    labels=[
        "Low (0-40)",
        "Medium (41-60)",
        "High (61-80)",
        "Very High (81-100)"
    ],
    include_lowest=True
)

score_band_analysis = (
    df_clean.groupby(
        "Lead_Score_Band",
        observed=False
    )
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum")
    )
)

score_band_analysis["Conversion_Rate_%"] = (
    score_band_analysis["Won_Leads"]
    / score_band_analysis["Leads"]
    * 100
)

print(
    score_band_analysis
    .round(2)
    .to_string()
)

LEAD QUALITY & ENGAGEMENT ANALYSIS

--- OVERALL METRICS ---
Average Lead Score: 59.37
Average Interactions: 5.02
Average Website Visits: 7.99
Average Email Opens: 3.99

--- WON VS NOT WON ---
        Leads  Avg_Lead_Score  Avg_Interactions  Avg_Website_Visits  Avg_Email_Opens  Avg_Satisfaction
Is_Won                                                                                                
False    4181           55.59              4.99                8.00             3.99              7.37
True      819           78.47              5.16                7.93             3.98              7.39

--- LEAD SCORE BY STAGE ---
             Leads  Avg_Lead_Score  Avg_Interactions  Avg_Website_Visits  Avg_Email_Opens
Lead_Stage                                                                               
Won            819           78.47              5.16                7.93             3.98
Negotiation    523           72.27              5.01                7.89             3.85
Proposa

In [14]:
# ============================================
# FINAL BUSINESS INSIGHTS SUMMARY
# ============================================

print("=" * 70)
print("FINAL CRM BUSINESS INSIGHTS")
print("=" * 70)

# -------------------------------------------------
# Overall KPIs
# -------------------------------------------------

total_customers = df_clean["Customer_ID"].nunique()
total_leads = len(df_clean)
won_leads = df_clean["Is_Won"].sum()
total_revenue = df_clean["Revenue"].sum()

conversion_rate = (
    won_leads / total_leads * 100
)

avg_revenue = df_clean["Revenue"].mean()
avg_lead_score = df_clean["Lead_Score"].mean()
avg_response_time = df_clean["Response_Time_Hours"].mean()
avg_satisfaction = df_clean["Customer_Satisfaction"].mean()

print("\n--- OVERALL KPIs ---")
print(f"Total Customers       : {total_customers:,}")
print(f"Total Leads           : {total_leads:,}")
print(f"Won Leads             : {won_leads:,}")
print(f"Total Revenue         : ${total_revenue:,.2f}")
print(f"Conversion Rate       : {conversion_rate:.2f}%")
print(f"Average Revenue       : ${avg_revenue:,.2f}")
print(f"Average Lead Score    : {avg_lead_score:.2f}")
print(f"Average Response Time : {avg_response_time:.2f} hours")
print(f"Average Satisfaction  : {avg_satisfaction:.2f}/10")


# -------------------------------------------------
# Top Segment by Revenue
# -------------------------------------------------

top_segment = (
    df_clean.groupby("Customer_Segment")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("\n--- TOP SEGMENT BY REVENUE ---")
print(top_segment.round(2).to_string())


# -------------------------------------------------
# Top Region by Revenue
# -------------------------------------------------

top_region = (
    df_clean.groupby("Region")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)

print("\n--- TOP REGION BY REVENUE ---")
print(top_region.round(2).to_string())


# -------------------------------------------------
# Top Region by Conversion
# -------------------------------------------------

region_conversion_final = (
    df_clean.groupby("Region")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum")
    )
)

region_conversion_final["Conversion_Rate_%"] = (
    region_conversion_final["Won_Leads"]
    / region_conversion_final["Leads"]
    * 100
)

print("\n--- TOP REGION BY CONVERSION ---")
print(
    region_conversion_final
    .sort_values("Conversion_Rate_%", ascending=False)
    .round(2)
    .to_string()
)


# -------------------------------------------------
# Top Lead Source
# -------------------------------------------------

source_conversion_final = (
    df_clean.groupby("Lead_Source")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum")
    )
)

source_conversion_final["Conversion_Rate_%"] = (
    source_conversion_final["Won_Leads"]
    / source_conversion_final["Leads"]
    * 100
)

print("\n--- LEAD SOURCE PERFORMANCE ---")
print(
    source_conversion_final
    .sort_values("Conversion_Rate_%", ascending=False)
    .round(2)
    .to_string()
)


# -------------------------------------------------
# Top Campaign
# -------------------------------------------------

campaign_conversion_final = (
    df_clean.groupby("Campaign")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum")
    )
)

campaign_conversion_final["Conversion_Rate_%"] = (
    campaign_conversion_final["Won_Leads"]
    / campaign_conversion_final["Leads"]
    * 100
)

print("\n--- CAMPAIGN PERFORMANCE ---")
print(
    campaign_conversion_final
    .sort_values("Conversion_Rate_%", ascending=False)
    .round(2)
    .to_string()
)


# -------------------------------------------------
# Sales Representative
# -------------------------------------------------

sales_rep_final = (
    df_clean.groupby("Assigned_Sales_Rep")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum")
    )
)

sales_rep_final["Conversion_Rate_%"] = (
    sales_rep_final["Won_Leads"]
    / sales_rep_final["Leads"]
    * 100
)

print("\n--- SALES REPRESENTATIVE PERFORMANCE ---")
print(
    sales_rep_final
    .sort_values("Revenue", ascending=False)
    .round(2)
    .to_string()
)


# -------------------------------------------------
# Product
# -------------------------------------------------

product_final = (
    df_clean.groupby("Product")
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum"),
        Revenue=("Revenue", "sum")
    )
)

product_final["Conversion_Rate_%"] = (
    product_final["Won_Leads"]
    / product_final["Leads"]
    * 100
)

print("\n--- PRODUCT PERFORMANCE ---")
print(
    product_final
    .sort_values("Revenue", ascending=False)
    .round(2)
    .to_string()
)


# -------------------------------------------------
# Lead Score Bands
# -------------------------------------------------

score_band_final = (
    df_clean.groupby(
        "Lead_Score_Band",
        observed=False
    )
    .agg(
        Leads=("Customer_ID", "count"),
        Won_Leads=("Is_Won", "sum")
    )
)

score_band_final["Conversion_Rate_%"] = (
    score_band_final["Won_Leads"]
    / score_band_final["Leads"]
    * 100
)

print("\n--- LEAD SCORE PERFORMANCE ---")
print(
    score_band_final
    .round(2)
    .to_string()
)


print("\n" + "=" * 70)
print("FINAL BUSINESS ANALYSIS COMPLETED")
print("=" * 70)

FINAL CRM BUSINESS INSIGHTS

--- OVERALL KPIs ---
Total Customers       : 5,000
Total Leads           : 5,000
Won Leads             : 819
Total Revenue         : $1,643,196.57
Conversion Rate       : 16.38%
Average Revenue       : $329.10
Average Lead Score    : 59.37
Average Response Time : 11.19 hours
Average Satisfaction  : 7.37/10

--- TOP SEGMENT BY REVENUE ---
Customer_Segment
Enterprise        639056.44
Corporate         578766.61
Small Business    242106.61
Consumer          183266.91

--- TOP REGION BY REVENUE ---
Region
Irbid      249681.17
Amman      233262.73
Jerash     207815.03
Madaba     207057.90
Aqaba      201185.88
Salt       194152.82
Karak      159311.71
Zarqa      153266.27
Unknown     37463.06

--- TOP REGION BY CONVERSION ---
         Leads  Won_Leads  Conversion_Rate_%
Region                                      
Jerash     642        123              19.16
Amman      660        116              17.58
Irbid      646        110              17.03
Karak      619  

In [15]:
# ============================================
# FINAL DATASET EXPORT
# ============================================

# Make a copy for final export
df_final = df_clean.copy()

# Export final CRM dataset
df_final.to_csv(
    "CRM_Final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CRM_Final.csv exported successfully.")
print(f"Rows: {df_final.shape[0]:,}")
print(f"Columns: {df_final.shape[1]}")

CRM_Final.csv exported successfully.
Rows: 5,000
Columns: 29


In [16]:
# ============================================
# FINAL EXPORT VALIDATION
# ============================================

final_check = pd.read_csv("CRM_Final.csv")

print("=" * 60)
print("FINAL EXPORT VALIDATION")
print("=" * 60)

print(f"\nRows: {final_check.shape[0]:,}")
print(f"Columns: {final_check.shape[1]:,}")

print(
    f"Duplicate rows: "
    f"{final_check.duplicated().sum():,}"
)

print(
    f"Duplicate Customer_ID: "
    f"{final_check['Customer_ID'].duplicated().sum():,}"
)

print(
    f"Missing cells: "
    f"{final_check.isna().sum().sum():,}"
)

print("\nLead Score invalid:",
      (
          (final_check["Lead_Score"] < 0) |
          (final_check["Lead_Score"] > 100)
      ).sum()
)

print(
    "Negative Revenue:",
    (final_check["Revenue"] < 0).sum()
)

print(
    "Negative Response Time:",
    (final_check["Response_Time_Hours"] < 0).sum()
)

print("\n" + "=" * 60)
print("FINAL CHECK COMPLETED")
print("=" * 60)

FINAL EXPORT VALIDATION

Rows: 5,000
Columns: 29
Duplicate rows: 0
Duplicate Customer_ID: 0
Missing cells: 1,488

Lead Score invalid: 0
Negative Revenue: 0
Negative Response Time: 0

FINAL CHECK COMPLETED


In [17]:
import os

print(os.path.abspath("CRM_Final.csv"))
print(os.path.exists("CRM_Final.csv"))

c:\Users\Omar\OneDrive - asu.edu.jo\Desktop\project911\AdventureWorks_Sales_Reporting\test project\CRM_Final.csv
True
